# Notebook 4 — Business Insights & Recommendations

## Executive Summary

- **Price tier is the strongest driver of rating** (ANOVA F=522.9, p≈0, η²=0.175 — price tier
  alone explains 17.5% of rating variance). Premium restaurants average 3.89 vs. 3.24 for
  low-price restaurants.
- **Table booking availability has a genuine, moderate positive association with rating**
  (Cohen's d=0.32, p=1.58e-22) and holds up as a real, actionable signal.
- **Online delivery's relationship with rating is not a simple "add it and ratings go up" story**
  — the raw effect is small and *negative* (Cohen's d=-0.16), but this is substantially
  **confounded by price tier**: delivery adoption is concentrated in low/mid-price restaurants,
  which already average lower ratings for unrelated reasons. Controlling for price tier, the
  within-tier effect of delivery is inconsistent in direction. **This directly reverses the
  original recommendation to "prioritize online delivery" — that recommendation is retracted.**
- Cuisine offerings are heavily concentrated: the top 5 cuisines account for ~51% of all listings,
  led by North Indian (18.5%).
- India's largest cities show real internal variation (Gurgaon > New Delhi > Noida > Faridabad),
  each with enough volume (n≥100) to be treated as reliable within-India findings.
- Every recommendation below is quantified and labeled correlational or causal — none are stated
  as fact without a supporting statistic from Notebooks 2–3.

**What changed from the original version of this notebook:** the original recommendation list
restated visual patterns as flat, unquantified claims and treated the online-delivery pattern as
causal. This version replaces that with effect sizes, confound-checking, and an explicit
correlation/causation label on every recommendation.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
PROJECT_ROOT = Path.cwd().parent
df = pd.read_csv(PROJECT_ROOT / "Dataset" / "cleaned_dataset.csv")
rated = df[df["Aggregate rating"] > 0].copy()
print("Rated restaurants:", rated.shape[0])

Rated restaurants: 7403

## Finding 1 — Price Tier (Strongest Driver, Causal Plausibility Reasonable)

In [1]:
groups = [g["Aggregate rating"].values for _, g in rated.groupby("Price range")]
f_stat, p_anova = stats.f_oneway(*groups)
grand_mean = rated["Aggregate rating"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((rated["Aggregate rating"] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total
print(f"ANOVA: F={f_stat:.3f}, p={p_anova:.2e}, eta-squared={eta_sq:.3f}")
print(rated.groupby("Price range")["Aggregate rating"].agg(["mean", "count"]))
print()
print("Price range distribution (% of all rated restaurants):")
print((rated["Price range"].value_counts(normalize=True) * 100).round(2))

ANOVA: F=522.892, p=3.34e-308, eta-squared=0.175
             mean  count
Price range
1            3.238885   2744
2            3.377167   2711
3            3.777276   1373
4            3.890957    575

Price range distribution (% of all rated restaurants):
Price range
1    37.07
2    36.62
3    18.55
4     7.77

**Interpretation:** Price tier explains 17.5% of total rating variance on its own — by far
the strongest single relationship found in this dataset. Plausible mechanism: higher-price
restaurants can invest more in food quality, service, and ambiance, all of which plausibly *cause*
better customer experience (unlike a simple demographic correlation). **Labeled: correlational
with reasonable causal plausibility, not proven causal** — we have no controlled experiment, only
observational data.

## Finding 2 — Table Booking (Real, Moderate, Actionable)

In [1]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * a.std() ** 2 + (nb - 1) * b.std() ** 2) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_std

tb_yes = rated.loc[rated["Has Table booking"] == 1, "Aggregate rating"]
tb_no = rated.loc[rated["Has Table booking"] == 0, "Aggregate rating"]
d_tb = cohens_d(tb_yes, tb_no)
print(f"Table Booking: mean(Yes)={tb_yes.mean():.3f} (n={len(tb_yes)}), "
      f"mean(No)={tb_no.mean():.3f} (n={len(tb_no)}), Cohen's d={d_tb:.3f}")
print()
print("Table booking adoption rate by price range (%):")
print((rated.groupby("Price range")["Has Table booking"].mean() * 100).round(1))

Table Booking: mean(Yes)=3.588 (n=1111), mean(No)=3.414 (n=6292), Cohen's d=0.316

Table booking adoption rate by price range (%):
Price range
1     0.0
2     8.1
3    45.4
4    46.3


**Interpretation:** A small-to-moderate effect (d=0.32) that is statistically robust. Note
it is also strongly tied to price tier (0% adoption at the lowest tier, ~46% at the top two
tiers) — so part of this association likely overlaps with the price-tier effect above rather than
being fully independent. **Labeled: correlational, partially confounded by price tier** — still
actionable (restaurants that can support table booking operationally should consider it), but not
a claim that adding booking alone will lift a low-price restaurant to premium-level ratings.

## Finding 3 — Online Delivery (Corrected Finding: Confounded, Not a Clean Positive)

This is the finding that required the most correction from the original notebook. The raw,
unconditional comparison suggested a positive relationship; the corrected, confound-checked
analysis below shows otherwise.

In [1]:
od_yes = rated.loc[rated["Has Online delivery"] == 1, "Aggregate rating"]
od_no = rated.loc[rated["Has Online delivery"] == 0, "Aggregate rating"]
d_od = cohens_d(od_yes, od_no)
print(f"Overall (unconditional): mean(Yes)={od_yes.mean():.3f}, mean(No)={od_no.mean():.3f}, Cohen's d={d_od:.3f}")
print()
print("Online delivery adoption rate by price range (%):")
print((rated.groupby("Price range")["Has Online delivery"].mean() * 100).round(1))
print()
print("Rating by Online Delivery, WITHIN each price range (controls for the price-tier confound):")
print(rated.groupby(["Price range", "Has Online delivery"])["Aggregate rating"].agg(["mean", "count"]).round(3))

Overall (unconditional): mean(Yes)=3.381, mean(No)=3.467, Cohen's d=-0.156

Online delivery adoption rate by price range (%):
Price range
1    23.6
2    45.9
3    29.8
4     9.2

Rating by Online Delivery, WITHIN each price range (controls for the price-tier confound):
                                  mean  count
Price range Has Online delivery
1           0                    3.232   2096
            1                    3.262    648
2           0                    3.414   1466
            1                    3.333   1245
3           0                    3.833    964
            1                    3.645    409
4           0                    3.887    522
            1                    3.930     53

**Interpretation:** The overall negative effect (d=-0.156) is real but small, and once you
condition on price tier, the direction is **not consistent** — delivery restaurants rate slightly
*higher* within the lowest and highest price tiers (3.262 vs 3.232; 3.930 vs 3.887), and lower
within the two middle tiers (3.333 vs 3.414; 3.645 vs 3.833). This is the classic signature of a
confounded aggregate statistic (adoption of delivery is heavily concentrated in price tier 2,
which pulls the unconditional average down).

**Corrected recommendation:** there is no clean, universal "add online delivery to improve your
rating" claim supportable by this data — in either direction. Any delivery-related recommendation
must be scoped to a specific price tier and treated as weak evidence, not a headline
recommendation. **This retracts the original notebook's #1 recommendation.**

## Finding 4 — Cuisine Concentration

In [1]:
cuisine_share = rated["Cuisines"].str.split(", ").explode().value_counts(normalize=True).head(5) * 100
print("Top 5 cuisines, % market share (rated restaurants):")
print(cuisine_share.round(2))
print(f"\nTop 5 combined share: {cuisine_share.sum():.1f}%")

Top 5 cuisines, % market share (rated restaurants):
Cuisines
North Indian    18.54
Chinese         13.42
Fast Food        9.60
Mughlai          4.88
Italian          4.46
Name: proportion, dtype: float64

Top 5 combined share: 50.9%

**Interpretation:** The top 5 cuisines account for just over half of all cuisine listings. This is descriptive market-structure information, not a rating driver by itself — no significance test is claimed here because none was run against cuisine type directly; that would require its own ANOVA/chi-square pass, flagged as a natural extension of this notebook rather than asserted without testing.

## Finding 5 — Sample-Size-Aware Geographic Segmentation (India Only)
Per the BRD, only countries/cities with n≥40 (per Notebook 2) are used for recommendations. Within India specifically, the 4 largest cities each have n≥100, making city-level comparison reliable at that scale.

In [1]:
india = rated[rated["Country Name"] == "India"]
city_stats = india.groupby("City")["Aggregate rating"].agg(["mean", "count"])
city_stats = city_stats[city_stats["count"] >= 100].sort_values("mean", ascending=False)
print(city_stats.round(3))

            mean  count
City
Gurgaon    3.331    890
New Delhi  3.297   4048
Noida      3.160    696
Faridabad  3.103    151

**Interpretation:** Within India's high-volume cities, Gurgaon narrowly leads, with Faridabad trailing. The gap (3.331 vs 3.103) is modest — roughly a quarter-point on a 5-point scale — and no significance test has been run on this specific comparison; it is reported as a directional, descriptive finding for now.

## Prioritized Recommendations

Each recommendation states its supporting statistic, correlation/causation label, and an
effort/impact rating (Low/Medium/High) so a stakeholder can decide what to act on first.

| # | Recommendation | Evidence | Label | Effort | Impact |
|---|---|---|---|---|---|
| 1 | Treat price positioning as the primary lever for expected rating — premium concepts should expect materially higher baseline ratings (3.89 vs 3.24 low-tier) | ANOVA F=522.9, η²=0.175 | Correlational, plausible causal mechanism | High (repositioning is a major business decision) | High |
| 2 | Offer table booking where operationally feasible | Cohen's d=0.32, p=1.58e-22 | Correlational, partially confounded with price tier | Medium | Medium |
| 3 | Do **not** treat online delivery adoption as a blanket rating lever in either direction | Confound-checked: direction reverses by price tier | Confounded — no clean directional claim supportable | Low (this is a "don't overcorrect" recommendation, not a build item) | Medium (prevents a bad decision) |
| 4 | Recognize North Indian/Chinese/Fast Food as the dominant competitive set (50.9% of listings) when evaluating market saturation for a new concept | Descriptive frequency count | Descriptive, not a rating driver | Low | Medium |
| 5 | Within India, treat Gurgaon vs. Faridabad-level city differences as modest, not decisive | Descriptive, n≥100 per city, no significance test run | Descriptive | Low | Low |

**Retracted from the original version:** "Restaurants should prioritize online delivery... as they
tend to receive higher ratings" — this claim does not survive a confound check and is replaced by
recommendation #3 above.

## Limitations Recap (see `docs/06_assumptions_and_limitations.md` for full detail)

- All findings above use the currency-normalized cost field and the 90.59%-India-skewed dataset;
  non-India findings are not reported here as recommendations, only India and overall patterns
  that had sufficient supporting sample size.
- No causal claim in this notebook is based on a controlled experiment — all are observational,
  labeled accordingly, and effect sizes are reported alongside significance so a reader can judge
  practical, not just statistical, significance.
- Cuisine-level rating differences were not statistically tested in this pass and are flagged as
  a natural next step rather than asserted.